<a href="https://colab.research.google.com/github/7omain/Uni-Coursework/blob/main/coursework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 6BUIS017W – Market Analysis and Stock Segmentation Using RFM

### Student ID: w2118173

### Data retrieved: 2026-10-21

In [20]:
!pip install -q yfinance==0.2.66

import os, sys, datetime as dt, requests
from io import StringIO
import pandas as pd
import yfinance as yf

RETRIEVAL_DATE = "2026-10-21"
START, END = "2021-01-01", "2025-01-01"
os.makedirs("data", exist_ok=True)

versions = {"python": sys.version.split()[0], "pandas": pd.__version__,
            "yfinance": yf.__version__, "retrieval_date": RETRIEVAL_DATE}
pd.Series(versions).to_csv("data/versions.csv")
print(versions)

{'python': '3.13.16', 'pandas': '2.2.3', 'yfinance': '0.2.66', 'retrieval_date': '2026-10-21'}


### Task 1.1.1

In [21]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0 (uni coursework)"})
resp.raise_for_status()

sp500 = pd.read_html(StringIO(resp.text))[0]
sp500["yahoo_ticker"] = sp500["Symbol"].str.replace(".", "-", regex=False)
sp500.to_csv(f"data/sp500_constituents_{RETRIEVAL_DATE}.csv", index=False)

print(len(sp500), "constituents retrieved on", RETRIEVAL_DATE)
sp500[["Symbol", "yahoo_ticker", "Security", "GICS Sector"]].head()



503 constituents retrieved on 2026-10-21


,Symbol,yahoo_ticker,Security,GICS Sector
0,MMM,MMM,3M,Industrials
1,AOS,AOS,A. O. Smith,Industrials
2,ABT,ABT,Abbott Laboratories,Health Care
3,ABBV,ABBV,AbbVie,Health Care
4,ACN,ACN,Accenture,Information Technology


### Task 1.1.2

In [22]:
sp500[sp500["Symbol"].str.contains(".", regex=False)][["Symbol", "yahoo_ticker"]]

,Symbol,yahoo_ticker
60,BRK.B,BRK-B
76,BF.B,BF-B


### Task 1.1.3

In [23]:
tickers = sp500["yahoo_ticker"].tolist() + ["^GSPC"]

raw = yf.download(tickers, start=START, end=END,
                  auto_adjust=False,
                  actions=True,
                  group_by="column", threads=True, progress=True)
print(raw.shape)
print(raw.columns.get_level_values(0).unique().tolist())

[**********************82%**************         ]  411 of 504 completedERROR:yfinance:Failed to get ticker 'RMD' reason: Failed to perform, curl: (28) Operation timed out after 10000 milliseconds with 0 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
[*********************100%***********************]  504 of 504 completed
ERROR:yfinance:
5 Failed downloads:
ERROR:yfinance:['HONA', 'VYLR', 'FDXF', 'SNDK', 'Q']: YFPricesMissingError('possibly delisted; no price data found  (1d 2021-01-01 -> 2025-01-01) (Yahoo error = "Data doesn\'t exist for startDate = 1609477200, endDate = 1735707600")')


(1005, 4022)
['Adj Close', 'Close', 'Dividends', 'High', 'Low', 'Open', 'Stock Splits', 'Volume']


### Task 1.1.4

In [24]:
failed = ["HONA", "VYLR", "FDXF", "SNDK", "Q"]
sp500[sp500["yahoo_ticker"].isin(failed)][["Symbol", "Security", "GICS Sector", "Date added"]]

,Symbol,Security,GICS Sector,Date added
192,FDXF,FedEx Freight,Industrials,2026-06-01
233,HONA,Honeywell Aerospace,Industrials,2026-06-29
382,Q,Qnity Electronics,Information Technology,2025-11-03
402,SNDK,Sandisk,Information Technology,2025-11-28
478,VYLR,Vylor,Consumer Staples,2026-10-01


### Task 1.1.5

In [25]:
long = (raw.stack(level=1, future_stack=True)
           .rename_axis(["Date", "Ticker"])
           .reset_index())
fname = f"data/prices_raw_{START}_{END}_retrieved_{RETRIEVAL_DATE}.csv.gz"
long.to_csv(fname, index=False)
print(long.shape)
print(round(os.path.getsize(fname) / 1e6, 1), "MB")
long.head()

(506520, 10)
16.6 MB


Price,Date,Ticker,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
0,2021-01-04,A,114.009132,118.639999,0.194,120.089996,117.110001,118.940002,0.0,2030700.0
1,2021-01-04,AAPL,125.632515,129.410004,0.000,133.610001,126.760002,133.520004,0.0,143301900.0
2,2021-01-04,ABBV,84.219124,105.410004,0.000,107.349998,103.860001,107.180000,0.0,9523400.0
3,2021-01-04,ABNB,139.149994,139.149994,0.000,151.005005,137.000000,150.990005,0.0,6409900.0
4,2021-01-04,ABT,97.269699,109.110001,0.000,110.250000,107.260002,109.529999,0.0,6192000.0


In [26]:
check = pd.read_csv(fname, parse_dates=["Date"])
print(check.shape)
check.head()

(506520, 10)


,Date,Ticker,Adj Close,Close,Dividends,High,Low,Open,Stock Splits,Volume
0,2021-01-04,A,114.009132,118.639999,0.194,120.089996,117.110001,118.940002,0.0,2030700.0
1,2021-01-04,AAPL,125.632515,129.410004,0.000,133.610001,126.760002,133.520004,0.0,143301900.0
2,2021-01-04,ABBV,84.219124,105.410004,0.000,107.349998,103.860001,107.180000,0.0,9523400.0
3,2021-01-04,ABNB,139.149994,139.149994,0.000,151.005005,137.000000,150.990005,0.0,6409900.0
4,2021-01-04,ABT,97.269699,109.110001,0.000,110.250000,107.260002,109.529999,0.0,6192000.0


### Task 1.1.6

In [29]:
adj = check.pivot(index="Date", columns="Ticker", values="Adj Close")
bench_days = adj["^GSPC"].dropna().index

coverage = pd.DataFrame({
    "first_date": adj.apply(lambda s: s.first_valid_index()),
    "last_date": adj.apply(lambda s: s.last_valid_index()),
    "n_days": adj.notna().sum(),
})
coverage["expected_days"] = len(bench_days)

def reason (row):
  if row.n_days == 0:                 return "no data returned"
  if row.first_date > bench_days[0]:  return f"listed later ({row.first_date.date()})"
  if row.last_date  < bench_days[-1]: return f"history ends early ({row.last_date.date()})"
  if row.n_days < len(bench_days):    return "missing trading days"
  return None

coverage["exclusion_reason"] = coverage.apply(reason, axis=1)

exclusions = (coverage[coverage.exclusion_reason.notna()]
              .drop(index="^GSPC", errors="ignore")
              .merge(sp500.set_index("yahoo_ticker")[["Security", "GICS Sector", "Date added"]],
                     left_index=True, right_index=True, how="left")
              .sort_values("first_date"))
exclusions.to_csv("data/exclusion_log_1_1.csv")

retained = coverage[coverage.exclusion_reason.isna()].drop(index="^GSPC", errors="ignore").index
print("Retained:", len(retained), "| Excluded:", len(exclusions))
exclusions

Retained: 487 | Excluded: 16


,first_date,last_date,n_days,expected_days,exclusion_reason,Security,GICS Sector,Date added
Ticker,,,,,,,,
EXE,2021-02-10,2024-12-31,979,1005,listed later (2021-02-10),Expand Energy,Energy,2025-03-24
COIN,2021-04-14,2024-12-31,936,1005,listed later (2021-04-14),Coinbase,Financials,2025-05-19
APP,2021-04-15,2024-12-31,935,1005,listed later (2021-04-15),AppLovin,Communication Services,2025-09-22
HOOD,2021-07-29,2024-12-31,862,1005,listed later (2021-07-29),Robinhood Markets,Financials,2025-09-22
CEG,2022-01-19,2024-12-31,742,1005,listed later (2022-01-19),Constellation Energy,Utilities,2022-02-02
GEHC,2022-12-15,2024-12-31,513,1005,listed later (2022-12-15),GE HealthCare,Health Care,2023-01-04
KVUE,2023-05-04,2024-12-31,418,1005,listed later (2023-05-04),Kenvue,Consumer Staples,2023-08-25
VLTO,2023-10-04,2024-12-31,313,1005,listed later (2023-10-04),Veralto,Industrials,2023-10-02
RDDT,2024-03-21,2024-12-31,197,1005,listed later (2024-03-21),Reddit,Communication Services,2026-08-18
